In [7]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

image_dir = Path.cwd() / "images"
image_files = ["lenna.png", "baboon.png", "barbara.png", "cat.png"]

# No downloads: the images are stored locally in the images/ folder.
# Just verify they are all there before continuing.
missing = [f for f in image_files if not (image_dir / f).exists()]
if missing:
    raise FileNotFoundError(
        "Missing images in the images/ folder: " + ", ".join(missing)
    )

print("Images ready:", ", ".join(image_files))

Images ready: lenna.png, baboon.png, barbara.png, cat.png


## 1. Getting an image into Python

OpenCV reads images as NumPy arrays. That means the image has a shape, a data type, and values that we can inspect just like any other array.

The first useful check is whether the image loaded successfully. Try the inspection commands below before moving on.

In [ ]:
image = cv2.imread("images/___")

print("shape:", image.___)
print("dtype:", image.___)
print("minimum pixel value:", image.___())
print("maximum pixel value:", image.___())

## 2. Displaying colors correctly

OpenCV stores color images in BGR order, while Matplotlib expects RGB. The image can look strange if we forget to convert between those two conventions.

Use the conversion constant that changes BGR into RGB.

In [ ]:
rgb_image = cv2.cvtColor(image, ___)
plt.imshow(rgb_image)
plt.axis("off")
plt.show()

## 3. Grayscale and color channels

A grayscale image has one intensity value per pixel instead of three color channels. After converting it, compare its shape with the original image.

The channel order in the original OpenCV image is blue, green, red.

In [ ]:
gray_image = cv2.cvtColor(image, ___)
print("gray shape:", gray_image.___)

blue = image[:, :, ___]
green = image[:, :, ___]
red = image[:, :, ___]

plt.figure(figsize=(12, 3))
for position, channel, title in [(1, blue, "Blue"), (2, green, "Green"), (3, red, "Red")]:
    plt.subplot(1, 3, position)
    plt.imshow(channel, cmap="gray")
    plt.title(title)
    plt.axis("off")
plt.show()

## 4. Copying, aliasing, and cropping

Assigning an array to a second variable does not create a second image. Both names can refer to the same data. Use `.copy()` when an edit should not change the original.

Cropping is just NumPy slicing: rows first, columns second, and channels last.

In [ ]:
image_copy = image.___()
image_copy[___:___, ___:___, :] = 0

crop = image[___:___, ___:___, :]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Original")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
plt.title("Crop")
plt.axis("off")
plt.show()

## 5. Flipping and drawing

`flipCode=0` flips vertically, `flipCode=1` flips horizontally, and `flipCode=-1` flips both directions.

In [ ]:
vertical = cv2.flip(image, ___)
horizontal = cv2.flip(image, ___)

annotated = image.copy()
cv2.rectangle(annotated, (50, 50), (200, 200), (0, 255, 0), 3)
cv2.putText(annotated, "Book?", (60, 240), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

plt.figure(figsize=(12, 4))
for position, picture, title in [(1, vertical, "Vertical"), (2, horizontal, "Horizontal"), (3, annotated, "Annotation")]:
    plt.subplot(1, 3, position)
    plt.imshow(cv2.cvtColor(picture, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
plt.show()

## 6. Scaling, translation, and rotation

Geometric transformations change where pixels appear or how many pixels an image contains. Keep track of the order of dimensions: NumPy reports `(rows, columns, channels)`, while OpenCV resize expects `(width, height)` when an explicit size is supplied.

In [ ]:
scaled = cv2.resize(image, None, fx=___, fy=___, interpolation=cv2.___)
print("original:", image.shape)
print("scaled:", scaled.shape)

rows, cols = image.shape[:2]
tx, ty = ___, ___
translation_matrix = np.float32([[1, 0, tx], [0, 1, ty]])
translated = cv2.warpAffine(image, translation_matrix, (cols + tx, rows + ty))

center = (cols // 2, rows // 2)
rotation_matrix = cv2.getRotationMatrix2D(center, ___, ___)
rotated = cv2.warpAffine(image, rotation_matrix, (cols, rows))

plt.figure(figsize=(12, 4))
for position, picture, title in [(1, scaled, "Scaled"), (2, translated, "Translated"), (3, rotated, "Rotated")]:
    plt.subplot(1, 3, position)
    plt.imshow(cv2.cvtColor(picture, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
plt.show()

## 7. Array arithmetic and noise

An image is an array, so arithmetic is applied element by element. Be careful with `uint8`: it stores values from 0 to 255, and arithmetic can wrap around instead of behaving like ordinary brightness.

In [ ]:
float_image = image.astype(___)
brighter = np.clip(float_image + ___, 0, ___).astype(np.uint8)

noise = np.random.normal(0, ___, image.shape).astype(np.float32)
noisy = np.clip(float_image + noise, 0, ___).astype(np.uint8)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(brighter, cv2.COLOR_BGR2RGB))
plt.title("Brighter")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(noisy, cv2.COLOR_BGR2RGB))
plt.title("With noise")
plt.axis("off")
plt.show()

## 8. Saving images and file paths

An image is just a file on disk, and `cv2.imwrite` writes a NumPy array back to one. The extension you choose (`.png`, `.jpg`, ...) selects the output format. The `os` module helps build full paths with `os.path.join`.

Fill in the blanks to print the full path of the loaded image and save it as a new `jpg` file.

In [ ]:
cwd = os.getcwd()
image_path = os.path.join(cwd, "images", "___")
print("full path:", image_path)

saved = cv2.imwrite("images/___", image)
print("image saved:", saved)

## 9. Loading a grayscale image from disk

Instead of converting to grayscale after loading, `imread` can decode the file directly as a single-channel image when the flag parameter is set to `cv2.IMREAD_GRAYSCALE`. Grayscale images must be shown with `cmap="gray"` in Matplotlib.

Fill in the blanks so `barbara.png` loads in grayscale, and print its shape to confirm it only has two dimensions.

In [ ]:
barbara_gray = cv2.imread("images/barbara.png", ___)
print("barbara_gray shape:", barbara_gray.___)

plt.imshow(barbara_gray, cmap="___")
plt.axis("off")
plt.show()

## 10. Isolating color channels

Setting a channel's values to zero removes that color from the image. In the BGR ordering, index 0 is blue, 1 is green, and 2 is red. To keep only the red channel, zero out indices 0 and 1.

Fill in the blanks to create red-only and blue-only versions of `baboon.png`. Work on copies so the original stays untouched.

In [ ]:
baboon = cv2.imread("images/baboon.png")

baboon_red = baboon.___()
baboon_red[:, :, 0] = ___
baboon_red[:, :, 1] = ___

baboon_blue = baboon.___()
baboon_blue[:, :, 1] = ___
baboon_blue[:, :, 2] = ___

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(baboon_red, cv2.COLOR_BGR2RGB))
plt.title("Red only")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(baboon_blue, cv2.COLOR_BGR2RGB))
plt.title("Blue only")
plt.axis("off")
plt.show()

## 11. Rotating by 90 and 180 degrees

Besides `getRotationMatrix2D`, OpenCV offers `cv2.rotate`, which rotates in exact 90-degree steps using built-in constants such as `cv2.ROTATE_90_CLOCKWISE`, `cv2.ROTATE_90_COUNTERCLOCKWISE`, and `cv2.ROTATE_180`.

Fill in the blanks to rotate `cat.png` clockwise by 90 degrees and by 180 degrees.

In [ ]:
cat = cv2.imread("images/cat.png")

clockwise = cv2.rotate(cat, ___)
half_turn = cv2.rotate(cat, ___)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(clockwise, cv2.COLOR_BGR2RGB))
plt.title("90 degrees clockwise")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(half_turn, cv2.COLOR_BGR2RGB))
plt.title("180 degrees")
plt.axis("off")
plt.show()

## 12. Final exercise

Use `baboon.png` for this exercise. Make a copy, crop a rectangular region, rotate the result, and display the original and edited images side by side. The earlier cells contain every operation you need, but you will need to choose sensible values for the crop and rotation.

In [ ]:
final_image = cv2.imread("images/___")
final_copy = final_image.___()
final_crop = final_copy[___:___, ___:___, :]
final_rows, final_cols = final_crop.shape[:2]
final_matrix = cv2.getRotationMatrix2D((final_cols // 2, final_rows // 2), ___, ___)
final_rotated = cv2.warpAffine(final_crop, final_matrix, (final_cols, final_rows))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(final_image, cv2.COLOR_BGR2RGB))
plt.title("Original")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(final_rotated, cv2.COLOR_BGR2RGB))
plt.title("Edited")
plt.axis("off")
plt.show()